# Jaida Calvin — NLP Project Update 2 Analysis
**This notebook performs new analysis of the saved Update 1 predictions. It does not retrain a model.**

Completed additions: paired error transitions, matched seven-group identity comparisons, paired template-bootstrap intervals, and score shifts. The original two-epoch DeHateBERT experiment remains the only completed model run. Cardiff and loss-function comparisons are still future work.

Run one cell at a time. A CPU runtime is enough. Existing repository inputs are pinned and checked by SHA-256; no model weights or raw offensive text are downloaded. Code and analysis were prepared with AI assistance. The outputs retained in this notebook were executed during preparation of Update 2.

## 1. Load the original saved predictions
When the notebook is run in the repository, it reads `results/`. Otherwise it downloads the same four small files from the Update 1 repository snapshot. A mismatch stops execution rather than silently analyzing different inputs.

In [1]:
from pathlib import Path
import hashlib, json, sys, urllib.request
import numpy as np
import pandas as pd
from IPython.display import display
INPUT = Path("results")
INPUT.mkdir(exist_ok=True)
BASE_URL = "https://raw.githubusercontent.com/JaiMayrie/nlp-hate-speech-false-positives/664a09f74661dc9fb89d62374715f7583f1d3265/results/"
EXPECTED = {'dehatebert_released_predictions.csv': 'f75bb66094131bae49128f3fce31a4bceda9de3c6f34819985f5a1e7d2110ce8', 'dehatebert_released_metrics.json': 'dcccce6b0ead1e462bf1891325058ebfa28203cea823bb7ab3d9363306491a07', 'dehatebert_finetuned_predictions.csv': '59c3e10d0041b86a20ec21947391ab5e958c255c3fc335e38f138a0635015de5', 'dehatebert_finetuned_metrics.json': '532ffdc5e894d6a97b2fc9f28c976ae8feb619d366e77d7eca4a072dbd661bf0'}
for name, expected in EXPECTED.items():
    path = INPUT / name
    if not path.exists():
        request = urllib.request.Request(BASE_URL + name, headers={"User-Agent": "NLP-Update2-analysis"})
        with urllib.request.urlopen(request, timeout=60) as response:
            data = response.read()
        if hashlib.sha256(data).hexdigest() != expected:
            raise ValueError(f"Downloaded input hash mismatch: {name}")
        path.write_bytes(data)
    if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
        raise ValueError(f"Existing input hash mismatch: {name}. Preserve it and use a fresh folder.")
print("Original inputs verified. No model training was performed.")
print("Python", sys.version.split()[0], "NumPy", np.__version__, "pandas", pd.__version__)


Original inputs verified. No model training was performed.
Python 3.13.5 NumPy 2.3.5 pandas 2.2.3


## 2. Load the analysis implementation
The code is embedded so this notebook works before or after the update2 folder is uploaded. It is the same implementation as `analyze_update2.py`.

In [2]:
ANALYSIS_SOURCE = '"""Reproducible Update 2 analysis of the saved Update 1 predictions.\n\nNo models are trained and no new neural inference is performed by this script.\nInputs are the original results/ CSV files; exports contain no raw text.\nRun from the repository root:\n    python update2/analyze_update2.py --input-dir results --output-dir update2\nRequires Python 3.10+, numpy, and pandas. AI-assisted implementation.\n"""\nfrom __future__ import annotations\nimport argparse\nimport hashlib\nimport json\nimport platform\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\n\nGROUPS = [\'women\',\'trans people\',\'gay people\',\'black people\',\'disabled people\',\'Muslims\',\'immigrants\']\nFUNCTIONS = [\'ident_neutral_nh\', \'ident_pos_nh\']\n\n\ndef require(condition: bool, message: str) -> None:\n    if not bool(condition):\n        raise ValueError(message)\n\n\ndef metrics(y: np.ndarray, pred: np.ndarray) -> dict:\n    tn=int(((y==0)&(pred==0)).sum()); fp=int(((y==0)&(pred==1)).sum())\n    fn=int(((y==1)&(pred==0)).sum()); tp=int(((y==1)&(pred==1)).sum())\n    p=tp/(tp+fp) if tp+fp else 0.; r=tp/(tp+fn) if tp+fn else 0.\n    f1_h=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.\n    f1_n=2*tn/(2*tn+fp+fn) if 2*tn+fp+fn else 0.\n    return dict(n=len(y),tn=tn,fp=fp,fn=fn,tp=tp,accuracy=(tp+tn)/len(y),\n                precision_hate=p,recall_hate=r,macro_f1=(f1_h+f1_n)/2,\n                fpr=fp/(fp+tn),fnr=fn/(fn+tp))\n\n\ndef run(input_dir: Path, output_dir: Path, seed: int=42, resamples: int=1000) -> dict:\n    require(resamples>0, \'Bootstrap resamples must be positive.\')\n    rd=output_dir/\'results\'; ed=output_dir/\'evidence\'\n    rd.mkdir(parents=True,exist_ok=True); ed.mkdir(parents=True,exist_ok=True)\n    checks=[]; hashes={}; dfs={}; computed=[]\n    cols=[\'id\',\'label\',\'functionality\',\'target_ident\',\'templ_id\',\'hate_score\',\'prediction\']\n    for condition in [\'released\',\'finetuned\']:\n        path=input_dir/f\'dehatebert_{condition}_predictions.csv\'\n        payload=path.read_bytes(); hashes[path.name]=dict(sha256=hashlib.sha256(payload).hexdigest(),\n            git_blob_sha1=hashlib.sha1(f\'blob {len(payload)}\\0\'.encode()+payload).hexdigest())\n        df=pd.read_csv(path)\n        require(set(cols)<=set(df.columns),f\'Missing columns: {path}\')\n        require(df.id.is_unique,f\'Duplicate case IDs: {path}\')\n        require(len(df)==3728,f\'Unexpected case count in {path}\')\n        require(df.label.isin([0,1]).all(),f\'Invalid labels: {path}\')\n        require(df.hate_score.between(0,1).all(),f\'Invalid probabilities: {path}\')\n        require(np.array_equal((df.hate_score>=0.5).astype(int),df.prediction),f\'Prediction threshold mismatch: {path}\')\n        dfs[condition]=df.sort_values(\'id\').reset_index(drop=True)\n        m=metrics(df.label.to_numpy(),df.prediction.to_numpy())\n        saved=json.loads((input_dir/f\'dehatebert_{condition}_metrics.json\').read_text())\n        for key,value in m.items():\n            if key in saved:\n                require(np.isclose(value,saved[key],rtol=1e-10,atol=1e-10),f\'Metric mismatch: {condition}/{key}\')\n        computed.append(dict(model=\'dehatebert\',condition=condition,threshold=0.5,**m))\n        checks.append(f\'{condition}: unique IDs, 3728 cases, legal labels/scores, threshold agreement, and saved overall metrics verified\')\n    identity=[\'id\',\'label\',\'functionality\',\'target_ident\',\'templ_id\']\n    pd.testing.assert_frame_equal(dfs[\'released\'][identity],dfs[\'finetuned\'][identity])\n    p=dfs[\'released\'].merge(dfs[\'finetuned\'],on=identity,suffixes=(\'_released\',\'_finetuned\'),validate=\'one_to_one\')\n    checks.append(\'One-to-one paired join preserves all case labels and metadata across the two conditions\')\n    pd.DataFrame(computed).to_csv(rd/\'overall_recheck.csv\',index=False)\n    trans=p.groupby([\'label\',\'prediction_released\',\'prediction_finetuned\']).size().reset_index(name=\'count\')\n    trans.to_csv(rd/\'prediction_transitions.csv\',index=False)\n    details=[]\n    for name,g in p.groupby(\'functionality\',sort=True):\n        y=g.label.to_numpy(); a=g.prediction_released.to_numpy(); b=g.prediction_finetuned.to_numpy()\n        require(len(set(y))==1,f\'Mixed labels for {name}\')\n        details.append(dict(functionality=name,label=int(y[0]),n=len(g),\n          correct_both=int(((a==y)&(b==y)).sum()),wrong_both=int(((a!=y)&(b!=y)).sum()),\n          corrected=int(((a!=y)&(b==y)).sum()),new_errors=int(((a==y)&(b!=y)).sum())))\n    pd.DataFrame(details).to_csv(rd/\'category_transitions.csv\',index=False)\n    p[\'score_change\']=p.hate_score_finetuned-p.hate_score_released\n    p.to_csv(rd/\'paired_cases.csv\',index=False)\n    shifts=[]\n    for label,g in p.groupby(\'label\'):\n        shifts.append(dict(label=int(label),n=len(g),scores_decreased=int((g.score_change<0).sum()),\n           scores_increased=int((g.score_change>0).sum()),scores_equal=int((g.score_change==0).sum()),\n           fraction_decreased=float((g.score_change<0).mean()),\n           released_mean=float(g.hate_score_released.mean()),finetuned_mean=float(g.hate_score_finetuned.mean()),\n           released_median=float(g.hate_score_released.median()),finetuned_median=float(g.hate_score_finetuned.median()),\n           median_paired_change=float(g.score_change.median())))\n    pd.DataFrame(shifts).to_csv(rd/\'score_shift_by_label.csv\',index=False)\n    subset=p[p.functionality.isin(FUNCTIONS)&(p.label==0)&p.target_ident.isin(GROUPS)].copy()\n    keys=[\'functionality\',\'templ_id\']\n    coverage=subset.groupby(keys).target_ident.agg(lambda x:set(x))\n    keep=coverage[coverage.apply(lambda x:x==set(GROUPS))].index\n    subset=subset.set_index(keys).loc[keep].reset_index()\n    require(len(subset)>0,\'No complete seven-group templates; do not report unmatched group FPRs as matched.\')\n    cell=subset.groupby(keys+[\'target_ident\'])[[\'prediction_released\',\'prediction_finetuned\']].mean().reset_index()\n    mats={c:cell.pivot(index=keys,columns=\'target_ident\',values=f\'prediction_{c}\').reindex(columns=GROUPS).sort_index()\n          for c in [\'released\',\'finetuned\']}\n    require(mats[\'released\'].index.equals(mats[\'finetuned\'].index),\'Template indices differ.\')\n    require(not any(m.isna().any().any() for m in mats.values()),\'Missing template-group cells.\')\n    n=len(mats[\'released\']); rng=np.random.default_rng(seed); draws=rng.integers(0,n,size=(resamples,n))\n    boot={c:m.to_numpy()[draws].mean(axis=1) for c,m in mats.items()}\n    summary=[]\n    for condition,m in mats.items():\n        point=m.mean(axis=0).to_numpy(); low,high=np.quantile(boot[condition],[.025,.975],axis=0)\n        for i,group in enumerate(GROUPS):\n            summary.append(dict(condition=condition,target_ident=group,templates=n,\n              cases=int((subset.target_ident==group).sum()),false_positives=int(subset.loc[subset.target_ident==group,f\'prediction_{condition}\'].sum()),\n              fpr=float(point[i]),bootstrap_95_low=float(low[i]),bootstrap_95_high=float(high[i])))\n    pd.DataFrame(summary).to_csv(rd/\'matched_identity_fpr.csv\',index=False)\n    gaps=[]\n    for c,m in mats.items():\n        point=m.mean(axis=0).to_numpy(); samples=boot[c].max(axis=1)-boot[c].min(axis=1)\n        low,high=np.quantile(samples,[.025,.975])\n        gaps.append(dict(condition=c,templates=n,fpr_max_minus_min=float(point.max()-point.min()),\n             bootstrap_95_low=float(low),bootstrap_95_high=float(high)))\n    pd.DataFrame(gaps).to_csv(rd/\'matched_group_gaps.csv\',index=False)\n    deltas=[]\n    bd=boot[\'finetuned\']-boot[\'released\']; low,high=np.quantile(bd,[.025,.975],axis=0)\n    point=(mats[\'finetuned\']-mats[\'released\']).mean(axis=0).to_numpy()\n    for i,group in enumerate(GROUPS):\n        deltas.append(dict(target_ident=group,templates=n,delta_fpr=float(point[i]),bootstrap_95_low=float(low[i]),bootstrap_95_high=float(high[i])))\n    pd.DataFrame(deltas).to_csv(rd/\'matched_fpr_changes.csv\',index=False)\n    subset.to_csv(rd/\'matched_template_cases.csv\',index=False)\n    for c,m in mats.items():\n        m.to_csv(rd/f\'matched_template_matrix_{c}.csv\')\n    checks.append(f\'Matched subset has {len(subset)} cases across {n} complete seven-group templates; equal template weighting used\')\n    checks.append(\'Paired bootstrap uses the same sampled template indices for all identities and both conditions\')\n    for condition in [\'released\',\'finetuned\']:\n        m=next(x for x in computed if x[\'condition\']==condition)\n        require(sum(x[\'count\'] for x in trans.to_dict(\'records\'))==3728,\'Transition total incorrect\')\n        require(m[\'tn\']+m[\'fp\']==1165 and m[\'fn\']+m[\'tp\']==2563,\'Class denominators changed\')\n    checks.append(\'Class denominators and transition totals reconcile with overall metrics\')\n    manifest=dict(analysis=\'Update 2 post-hoc analysis of Update 1 saved predictions; no new training or inference\',\n        created_utc=datetime.now(timezone.utc).isoformat(),bootstrap_seed=seed,bootstrap_resamples=resamples,\n        bootstrap_unit=\'whole (functionality, templ_id) with all seven identities kept together\',\n        bootstrap_method=\'non-stratified percentile intervals; empirical template resampling, not population coverage\',\n        matched_functions=FUNCTIONS,matched_templates=n,matched_cases=len(subset),\n        caveat=\'All-zero empirical errors give degenerate [0,0] bootstrap intervals; they do not imply zero population risk.\',\n        environment={\'python\':platform.python_version(),\'numpy\':np.__version__,\'pandas\':pd.__version__},\n        input_hashes=hashes,verification_checks=checks,\n        unchanged=\'original predictions, labels, training settings, main threshold and Update 1 model experiment\')\n    (ed/\'analysis_audit.json\').write_text(json.dumps(manifest,indent=2)+\'\\n\',encoding=\'utf-8\')\n    print(pd.DataFrame(summary).to_string(index=False))\n    print(\'\\nTransitions:\\n\'+trans.to_string(index=False))\n    print(\'\\n\'+json.dumps(manifest,indent=2))\n    return manifest\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--input-dir\',type=Path,default=Path(\'results\'))\n    parser.add_argument(\'--output-dir\',type=Path,default=Path(\'update2\'))\n    parser.add_argument(\'--seed\',type=int,default=42)\n    parser.add_argument(\'--resamples\',type=int,default=1000)\n    args=parser.parse_args()\n    run(args.input_dir,args.output_dir,args.seed,args.resamples)\n'
OUTPUT = Path("update2")
OUTPUT.mkdir(exist_ok=True)
(OUTPUT / "analyze_update2.py").write_text(ANALYSIS_SOURCE, encoding="utf-8")
import importlib.util
spec = importlib.util.spec_from_file_location("update2_analysis", OUTPUT / "analyze_update2.py")
analysis = importlib.util.module_from_spec(spec)
spec.loader.exec_module(analysis)
print("Analysis implementation loaded.")


Analysis implementation loaded.


## 3. Compute the new analyses
The bootstrap resamples whole templates, keeping all identities and both model versions together. Its intervals describe variation across the observed templates, not population-wide error or training-seed uncertainty.

In [3]:
audit = analysis.run(INPUT, OUTPUT, seed=42, resamples=1000)
print("Update 2 analysis complete. These are new analyses of the original run, not a new training experiment.")


condition    target_ident  templates  cases  false_positives      fpr  bootstrap_95_low  bootstrap_95_high
 released           women         45     45                1 0.022222          0.000000           0.066667
 released    trans people         45     45                1 0.022222          0.000000           0.066667
 released      gay people         45     45                0 0.000000          0.000000           0.000000
 released    black people         45     45                0 0.000000          0.000000           0.000000
 released disabled people         45     45                0 0.000000          0.000000           0.000000
 released         Muslims         45     45                1 0.022222          0.000000           0.066667
 released      immigrants         45     45                7 0.155556          0.044444           0.266667
finetuned           women         45     45                0 0.000000          0.000000           0.000000
finetuned    trans people         45 

## 4. Read the results
“Released” and “finetuned” refer to the original Update 1 model versions. Zero errors on the narrow matched subset do not mean a model is unbiased; compare the full-benchmark false-negative rate as well.

In [4]:
for name in ["overall_recheck.csv", "prediction_transitions.csv", "matched_identity_fpr.csv", "matched_group_gaps.csv", "score_shift_by_label.csv"]:
    print("\n" + name)
    display(pd.read_csv(OUTPUT / "results" / name))



overall_recheck.csv


,model,condition,threshold,n,tn,fp,fn,tp,accuracy,precision_hate,recall_hate,macro_f1,fpr,fnr
0,dehatebert,released,0.5,3728,1005,160,1983,580,0.425161,0.783784,0.226297,0.417592,0.137339,0.773703
1,dehatebert,finetuned,0.5,3728,1109,56,2392,171,0.343348,0.753304,0.066719,0.298967,0.048069,0.933281



prediction_transitions.csv


,label,prediction_released,prediction_finetuned,count
0,0,0,0,981
1,0,0,1,24
2,0,1,0,128
3,0,1,1,32
4,1,0,0,1949
5,1,0,1,34
6,1,1,0,443
7,1,1,1,137



matched_identity_fpr.csv


,condition,target_ident,templates,cases,false_positives,fpr,bootstrap_95_low,bootstrap_95_high
0,released,women,45,45,1,0.022222,0.000000,0.066667
1,released,trans people,45,45,1,0.022222,0.000000,0.066667
2,released,gay people,45,45,0,0.000000,0.000000,0.000000
3,released,black people,45,45,0,0.000000,0.000000,0.000000
4,released,disabled people,45,45,0,0.000000,0.000000,0.000000
5,released,Muslims,45,45,1,0.022222,0.000000,0.066667
6,released,immigrants,45,45,7,0.155556,0.044444,0.266667
7,finetuned,women,45,45,0,0.000000,0.000000,0.000000
8,finetuned,trans people,45,45,0,0.000000,0.000000,0.000000
9,finetuned,gay people,45,45,0,0.000000,0.000000,0.000000



matched_group_gaps.csv


,condition,templates,fpr_max_minus_min,bootstrap_95_low,bootstrap_95_high
0,released,45,0.155556,0.044444,0.266667
1,finetuned,45,0.000000,0.000000,0.000000



score_shift_by_label.csv


,label,n,scores_decreased,scores_increased,scores_equal,fraction_decreased,released_mean,finetuned_mean,released_median,finetuned_median,median_paired_change
0,0,1165,1057,108,0,0.907296,0.178593,0.084291,0.052377,0.012955,-0.022541
1,1,2563,2364,199,0,0.922357,0.253421,0.106997,0.081967,0.015679,-0.041922


## 5. Save a reproduction archive
This optional cell was not needed to prepare the supplied report. Run it after reproducing the analysis, then save the executed notebook with File → Download → Download .ipynb. Keep the original Update 1 notebook and results unchanged.

In [ ]:
import shutil
archive = shutil.make_archive("Jaida_Update2_Reproduced_Analysis", "zip", root_dir=OUTPUT)
print("Saved:", archive)
try:
    from google.colab import files
except ImportError:
    print("Outside Colab: the ZIP is in the working folder.")
else:
    files.download(archive)
